<div style="background: linear-gradient(120deg, #0b1f4b 0%, #173d7a 38%, #2a78d6 72%, #eb6834 100%); padding: 32px 34px 28px 34px; border-radius: 14px; color: #ffffff;">
<div style="font-size: 12px; letter-spacing: 3px; text-transform: uppercase; opacity: 0.85;">Playground Series S6E11 &middot; Predicting Credit Card Fraud &middot; ROC-AUC</div>
<div style="font-size: 36px; font-weight: 800; line-height: 1.12; margin-top: 10px;">How much of the public top is real?</div>
<div style="font-size: 17px; margin-top: 8px; opacity: 0.95;">Public ladders, their noise, the winner's curse, and the nested CV that tells you which steps will survive the private 80%</div>
<div style="margin-top: 18px;">
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">public LB = 20% of test</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">private = 80%</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">nested CV</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px; margin-right: 6px;">every number computed here</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px; margin-right: 6px;">public inputs only, SHA-256 checked</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px;">CPU, ~10 min</span>
</div>
</div>

The top of the public board is a dense band of files a few hundred-thousandths apart. Many of them are **ladders**:
a strong public file plus small steps, each kept because the public score went up by one or two points
(1 point = 0.00001 AUC). But the public score is computed on only ~60,000 rows with about a thousand frauds. This
notebook measures, with public files only, **how big the noise of one such step is, how much of a ladder's gain
nested CV confirms, and how much of the top band is likely to move on the private 80%**. Its `submission.csv` is the credited public 0.95354 ladder file, rebuilt here; it
takes no side: the ladder authors themselves list every probe and check their steps on CV, which is what makes this
analysis possible.

<div style="background: #eef4fc; border-left: 5px solid #2a78d6; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Contents</b><br>
<a href="#ack">0. Acknowledgements and who did what</a><br>
<a href="#hero">The whole story in one figure</a><br>
<a href="#s1">1. The competition in five minutes</a> &mdash; data, AUC, how big the public and private parts are<br>
<a href="#s2">2. How noisy is the public LB?</a> &mdash; one file versus the difference of two files, and a one-line rule<br>
<a href="#s3">3. Anatomy of a ladder</a> &mdash; every public rung against its noise and against nested CV<br>
<a href="#s4">4. The winner's curse</a> &mdash; why the best of many probes looks better than it is<br>
<a href="#s5">5. CV can be tuned too</a> &mdash; early stopping on the scored fold, selection on the reported folds, tests inside one fold<br>
<a href="#s6">6. What did not work</a><br>
<a href="#s7">7. What to expect on private, and how to pick two final files</a><br>
<a href="#s8">8. Full credits and licences</a>
</div>

If this is useful to you, an upvote helps others find it. Corrections are very welcome in the comments.

<a id="ack"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">0 &middot; Acknowledgements and who did what</div>

This analysis stands entirely on other people's public work. Thank you in particular to **goodpjw2008**, whose
notebook series published a nested OOF of a top-band stack, a 70-member OOF bank on one shared split, and ladder
notebooks that list **every probe with its public score**, including the ones that lost; to **Areeb Ali**, whose
*Three Ladders* notebooks publish the full version history and a CV check of each step; and to **amanatar**,
**Moran Z**, **megayak**, **masayakawamata**, **daoviet** and **Chris Deotte** for the files and ideas the ladders are
built from. Without this level of openness nobody could check the public board against CV. The full list with links
and licences is in [section 8](#s8). If we misattributed anything, please tell us in the comments.

<table style="font-size: 13px; border-collapse: collapse;">
<tr style="background: #eef4fc;"><th style="text-align: left; padding: 5px 8px;">Who</th><th style="text-align: left; padding: 5px 8px;">What this notebook uses</th></tr>
<tr><td style="padding: 5px 8px;"><b>goodpjw2008</b></td><td style="text-align: left; padding: 5px 8px;">the nested OOF of the <a href="https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298">Part 12 wide LM stack</a> (our stand-in for a test set), the <a href="https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank">OOF bank</a>, the <a href="https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-two-gpt-2-seeds-lb-0-95354">LM Ladder</a> probe tables, the seed-7 LM and GPT-2 seed-2026 predictions, and the <code>find()</code> / logit-rank conventions</td></tr>
<tr><td style="padding: 5px 8px;"><b>Areeb Ali</b> (areebbb)</td><td style="text-align: left; padding: 5px 8px;"><a href="https://www.kaggle.com/code/areebbb/s6e11-three-ladders-two-seeds-lb-0-95354">Three Ladders</a>: version history v9&ndash;v20 with public scores, the probe table and the CV-evidence table</td></tr>
<tr><td style="padding: 5px 8px;"><b>amanatar</b>, <b>Moran Z</b></td><td style="text-align: left; padding: 5px 8px;">the <a href="https://www.kaggle.com/datasets/amanatar/s6e11-strike10-podium-apex-095340">Strike 10</a> and <a href="https://www.kaggle.com/datasets/moranzavdi/s6e11-strike9-lb-095330">Strike 9</a> files every ladder is built on</td></tr>
<tr><td style="padding: 5px 8px;"><b>megayak</b>, <b>masayakawamata</b>, <b>daoviet</b></td><td style="text-align: left; padding: 5px 8px;"><a href="https://www.kaggle.com/datasets/megayak/s6e11-target-first-lm">target-first LM</a>, <a href="https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof">target-first GPT-2</a> and <a href="https://www.kaggle.com/datasets/daoviet/s6e11-gp-dvae-orthogonal-apex">GP + DVAE</a> OOF and test predictions</td></tr>
<tr><td style="padding: 5px 8px;"><b>this notebook</b></td><td style="text-align: left; padding: 5px 8px;">the noise measurements, the one-line noise rule, the winner's-curse simulations, the CV-tuning demos and all figures (asteriospanoras)</td></tr>
</table>

<a id="setup"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">Setup &middot; public inputs, checked byte for byte</div>

Every input is a public file: the competition data, public OOF/test predictions and two public ladder base files.
Each one is found anywhere under `/kaggle/input` by name and checked by its SHA-256 (the `find()` idea is
goodpjw2008's). The notebook runs on CPU in about ten minutes and never trains a big model.

In [ ]:
# Imports, input discovery by SHA-256, the shared scale for predictions, and a fast AUC for repeated random splits.
import os, time, hashlib, warnings, math
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import rankdata, norm
from scipy.special import logit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
T0 = time.time()
def log(*a): print(f"[{(time.time() - T0) / 60:4.1f} min]", *a, flush=True)

INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("KAGGLE_WORKING", "/kaggle/working"))
_sha = {}
def sha256(p):
    p = Path(p)
    if p not in _sha: _sha[p] = hashlib.sha256(p.read_bytes()).hexdigest()
    return _sha[p]

def find(names, digest=None, inside=None, required=True):
    """A file anywhere under /kaggle/input, by (one of its) names and, if given, its SHA-256.
    If no file has the expected hash but one has the name, use it and say so (the author may have updated it)."""
    names = [names] if isinstance(names, str) else names
    hits = [p for n in names for p in sorted(INPUT.rglob(n)) if inside is None or inside in str(p)]
    good = [p for p in hits if digest is None or sha256(p) == digest]
    if good: return good[0]
    if hits:
        print(f"  NOTE: {names[0]} found but its SHA-256 differs from the one this notebook was written with; numbers may differ slightly.")
        return hits[0]
    if required: raise FileNotFoundError(names)
    print(f"  NOTE: {names[0]} not attached; the cells that need it are skipped.")
    return None

def L(v):
    """Logit rank: AUC only sees order, so every prediction goes on one scale (as the public ladders do)."""
    return logit((rankdata(v) - 0.5) / len(v))

def load(names, digest, required=True):
    p = find(names, digest, required=required)
    return None if p is None else np.load(p, allow_pickle=False).astype(np.float64)

def auc_prep(score):
    """Pre-sort a score once: the negatives in score order, and for every positive the number of negatives below it."""
    o = np.argsort(score, kind="stable"); yo = y[o]
    neg_o = o[~yo].astype(np.int32); pos_rows = o[yo].astype(np.int32)
    kpos = np.cumsum(~yo)[yo].astype(np.int32)
    return neg_o, pos_rows, kpos

def auc_counts(prep, counts, denom):
    """AUC on a bootstrap sample (counts[i] = how often row i was drawn): O(n) per sample, no sort (ties ignored)."""
    neg_o, pos_rows, kpos = prep
    cn = np.concatenate(([0], np.cumsum(counts[neg_o], dtype=np.int64)))
    return float((counts[pos_rows].astype(np.int64) * cn[kpos]).sum()) / denom

def auc_sort(yr, s):
    """Plain AUC by one sort (Mann-Whitney), ~3x faster than sklearn on 700k rows; ties ignored."""
    idx = np.flatnonzero(yr[np.argsort(s, kind="stable")]); npos = len(idx)
    return (idx.sum() - npos * (npos - 1) / 2) / (npos * (len(yr) - npos))

COMP = "playground-series-s6e11"
train = pd.read_csv(find("train.csv", inside=COMP))
test_ids = pd.read_csv(find("test.csv", inside=COMP), usecols=["id"])["id"].to_numpy()
sample = pd.read_csv(find("sample_submission.csv", inside=COMP))
y = train["is_fraud"].to_numpy().astype(bool)
FOLDS = [b for _, b in StratifiedKFold(5, shuffle=True, random_state=42).split(y, y)]   # the split every public OOF uses
N, NT = len(y), len(test_ids)
log(f"train {N:,} rows, fraud rate {y.mean():.4%}; test {NT:,} rows")

In [ ]:
# One style for every figure: a colorblind-checked palette with fixed roles, recessive axes, titles that state the
# takeaway, KPI tiles, and the helpers that place text without clutter.
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator
INK, INK2, MUTED, RULE, SURF, PAGE, NAVY = "#0b0b0b", "#52514e", "#8a8984", "#d6d5d0", "#fcfcfb", "#f9f9f7", "#0b1f4b"
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
PUB, PRIV, CV, BAND = C[0], C[1], C[2], "#e6e5df"      # fixed roles: public = blue, private = orange, CV = aqua
CVTXT = "#0f7a55"                                        # darker aqua for text next to aqua marks
plt.rcParams.update({"figure.facecolor": PAGE, "axes.facecolor": SURF, "savefig.facecolor": PAGE, "axes.edgecolor": "#c3c2b7",
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True,
                     "grid.color": "#e8e7e3", "grid.linewidth": 0.7, "axes.spines.top": False, "axes.spines.right": False,
                     "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11.5, "axes.titleweight": "bold",
                     "axes.titlecolor": INK, "axes.titlelocation": "left", "axes.titlepad": 10, "lines.linewidth": 2,
                     "legend.frameon": False, "axes.axisbelow": True, "figure.dpi": 110, "xtick.labelsize": 9, "ytick.labelsize": 9})
E5 = FuncFormatter(lambda v, _: f"{v:+g}" if v else "0")    # axis ticks in units of 1e-5 AUC

def headline(fig, title, sub=None, top_pad=0.0):
    """Figure title that states the takeaway (left aligned), with an optional grey subtitle."""
    h = fig.get_figheight()
    fig.text(0.012, 1 - 0.12 / h, title, ha="left", va="top", fontsize=14, fontweight="bold", color=INK)
    if sub: fig.text(0.012, 1 - 0.46 / h, sub, ha="left", va="top", fontsize=10, color=INK2)
    fig.tight_layout(rect=(0, 0, 1, 1 - ((0.80 if sub else 0.50) + top_pad) / h))

def tile(fig, rect, label, value, note, color=NAVY):
    """A KPI tile: label, big value, small note, with a coloured accent bar."""
    ax = fig.add_axes(rect); ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=0.08", transform=ax.transAxes,
                                facecolor=SURF, edgecolor="#e6e5df", lw=1.2))
    ax.add_patch(FancyBboxPatch((0, 0.10), 0.018, 0.80, boxstyle="round,pad=0,rounding_size=0.01", transform=ax.transAxes,
                                facecolor=color, edgecolor="none"))
    ax.text(0.07, 0.76, label, transform=ax.transAxes, fontsize=9.5, color=INK2, va="center")
    ax.text(0.07, 0.45, value, transform=ax.transAxes, fontsize=20, fontweight="bold", color=INK, va="center")
    ax.text(0.07, 0.15, note, transform=ax.transAxes, fontsize=8.5, color=MUTED, va="center")

def panel_title(ax, title, sub=None):
    ax.set_title(title, loc="left", fontsize=11.5, fontweight="bold", color=INK, pad=22 if sub else 8)
    if sub: ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=9, color=INK2, va="bottom")

def zero_line(ax, axis="y"):
    (ax.axhline if axis == "y" else ax.axvline)(0, color="#b9b8b3", lw=1, zorder=1)

**How the numbers below are made.** The test labels are hidden, so we use the next best thing: the **nested OOF of
goodpjw2008's public wide LM stack** (Part 12, AUC 0.9526 on 699,960 training rows). It is a real top-band
prediction with honest out-of-fold scores. We draw hundreds of random public parts (~60k rows) and private parts
(~240k rows) from it, as two parts of a fresh test set would be drawn, and watch how scores move. Every ladder step that has public OOF files
behind it is applied to the same OOF, exactly as the ladder authors did in their own CV checks.

In [ ]:
# Load the public OOF predictions (train side). W = the nested OOF of the public wide LM stack.
OOF_FILES = {   # key: (file name(s), SHA-256, who published it)
    "W":   ("wide_lm_stack_nested.npy", "69375871698f1adf2b4c666fc7f5180edf00ac6836668353b8e7e4ca43eb2722", "goodpjw2008, wide LM stack (nested OOF)"),
    "LM":  (["lm_oof.npy", "megalm_oof_sha181c.npy"], "181ceac8f3a48fcf9a1702e1222ee13178f0688a28c1aff22c8f881532ea981a", "megayak, target-first LM"),
    "L7F": ("lm7_oof_5fold.npy", "c56d6b7282e7ea8c4c5927ac27e8b286d539ffbd66d5795564323c8b92b65242", "goodpjw2008, compact LM seed 7"),
    "GPD": ("oof_ours.npy", "91be476ddb00fe1232ba98b9a30f3c6215e81769499e3594058e4e99703c706a", "daoviet, GP + DVAE"),
    "MLP": ("oof_gbn_bag5_w512.npy", "e8ef910425641b7be8f3bf8f964b1637e9fa39937a772df56ef0cb660d3776f3", "goodpjw2008, OOF bank MLP bag"),
    "GB":  ("oof_gb_bag5.npy", "4b4bc47571ee389a54a7e70d6ac2d716207c329a1ca619a37e577de1f30aea40", "goodpjw2008, OOF bank GB bag"),
    "MS":  ("oof.npy", "3468aaaeb7851739ec0de1e06b780da6d2777fcaeef638a6852c56e68e67a206", "masayakawamata, target-first GPT-2"),
    "G26": ("gpt2_s2026_oof_5fold.npy", "4a902ecd3d725d21123f5b6239ac2190a9fd94f3de37e3157be391fcf3fe996a", "goodpjw2008, GPT-2 seed 2026"),
}
O = {k: L(load(n, h)) for k, (n, h, _) in OOF_FILES.items()}
W = O["W"]
def cv(s): return np.mean([roc_auc_score(y[b], s[b]) for b in FOLDS])
def folds_up(s, base): return sum(roc_auc_score(y[b], s[b]) > roc_auc_score(y[b], base[b]) for b in FOLDS)
CV_W = cv(W)
for k in O: print(f"  {k:4s} OOF AUC {roc_auc_score(y, O[k]):.6f}   {OOF_FILES[k][2]}")
log(f"wide LM stack, nested CV {CV_W:.6f}")

# The OOF bank's 20 'core' members (one per recipe), used as extra probe directions and for the hill-climbing demo.
bank_dir = find("manifest.csv", inside="oof-bank").parent
bank_man = pd.read_csv(bank_dir / "manifest.csv")
CORE = [n for n in bank_man.loc[bank_man["core"], "name"] if (bank_dir / f"oof_{n}.npy").exists()]
BANK = {n: L(np.load(bank_dir / f"oof_{n}.npy", allow_pickle=False).astype(np.float64)) for n in CORE}
log(f"OOF bank: {len(BANK)} core members loaded")

In [ ]:
# Every public probe that has OOF files behind it, applied to W as a move  W + w * direction.
# 'pub' = the public-LB change the authors report for that single step (x 1e-5), with the source.
MOVES = [  # (short name, direction on the OOF, reported public change, source)
    ("+0.12 megayak LM",          0.12 * (O["LM"] - W),                       +1, "goodpjw2008 v1: 0.95342 -> 0.95343"),
    ("+0.06 seed-7 LM",           0.06 * (O["L7F"] - W),                      +2, "goodpjw2008 v1: 0.95343 -> 0.95345"),
    ("+0.12 public GPT-2",        0.12 * (O["MS"] - W),                      -14, "goodpjw2008 v1 probe: -14 points"),
    ("+0.05 GP + DVAE",           0.05 * (O["GPD"] - W),                      +3, "daoviet: 0.95348 -> 0.95351"),
    ("LM weight 0.12 -> 0.09",   -0.03 * (O["LM"] - W),                       +1, "Areeb Ali v19: 0.95347 -> 0.95348"),
    ("more of both LMs",          0.04 * (O["LM"] - W) + 0.03 * (O["L7F"] - W), -2, "Areeb Ali probe: 0.95347 -> 0.95345"),
    ("less seed-7 LM",           -0.02 * (O["L7F"] - W),                       0, "Areeb Ali probe: 0.95347 -> 0.95347"),
    ("GPT-2 seed swap 0.06",      0.06 * (O["G26"] - O["MS"]),                +2, "goodpjw2008 v2: 0.95352 -> 0.95354"),
    ("GPT-2 seed swap 0.10",      0.10 * (O["G26"] - O["MS"]),                +2, "goodpjw2008 v2 probe: 0.95354"),
]
TREE = -0.10 * (O["MLP"] - W) - 0.05 * (O["GB"] - W)        # Areeb Ali's full tree move (v15); no clean single-step public number
# A family of plausible probes: a small step towards (or away from) each bank member and each public LM.
rng = np.random.default_rng(11)
FAMILY = []
for name, X in list(BANK.items()) + [("megayak LM", O["LM"]), ("seed-7 LM", O["L7F"]), ("public GPT-2", O["MS"]),
                                     ("GPT-2 seed 2026", O["G26"]), ("GP + DVAE", O["GPD"])]:
    w = rng.choice([-0.10, -0.06, -0.03, 0.03, 0.06, 0.10])
    FAMILY.append((f"{w:+.2f} {name}", w * (X - W)))
SCORES = [("base", np.zeros(N))] + [(m[0], m[1]) for m in MOVES] + [("tree move", TREE)] + FAMILY
log(f"{len(SCORES) - 1} moves to score")

RHO, TRUE, UP = {}, {}, {}
RW = rankdata(W)
for name, d in SCORES[1:]:
    s = W + d
    RHO[name] = np.corrcoef(rankdata(s), RW)[0, 1]
    TRUE[name] = (cv(s) - CV_W) * 1e5
    UP[name] = folds_up(s, W)
log("nested-CV change of every move computed")

In [ ]:
# Random public/private splits: the public part is a bootstrap sample of NPUB rows from the OOF, the private part an
# independent bootstrap sample of the other NT - NPUB rows (independent draws, as two parts of a fresh test set would be).
R = int(os.environ.get("NB_DRAWS", 300))
NPUB = int(round(0.2 * NT))
preps = [auc_prep(W + d) for _, d in SCORES]
AUC = np.zeros((len(SCORES), R, 2))
rng = np.random.default_rng(2026)
for r in range(R):
    cp = np.bincount(rng.integers(0, N, NPUB), minlength=N).astype(np.int32)
    cq = np.bincount(rng.integers(0, N, NT - NPUB), minlength=N).astype(np.int32)
    dp = float(cp[y].sum()) * float(cp[~y].sum()); dq = float(cq[y].sum()) * float(cq[~y].sum())
    for i, pr in enumerate(preps):
        AUC[i, r, 0] = auc_counts(pr, cp, dp); AUC[i, r, 1] = auc_counts(pr, cq, dq)
    if r % 100 == 99: log(f"  {r + 1} of {R} splits")
del preps
DELTA = (AUC[1:] - AUC[:1]) * 1e5           # (move, draw, public/private) change vs the base file, in 1e-5
NAMES = [n for n, _ in SCORES[1:]]
SD_PUB = DELTA[:, :, 0].std(1); SD_PRV = DELTA[:, :, 1].std(1)
AGREE = (np.sign(DELTA[:, :, 0]) == np.sign(DELTA[:, :, 1])).mean(1)
ONE_RHO = np.array([1 - RHO[n] for n in NAMES])
K_RULE = np.exp(np.mean(np.log(SD_PUB * 1e-5 / np.sqrt(ONE_RHO))))   # sigma_public(delta) ~= K_RULE * sqrt(1 - rho)
SD_BASE_PUB, SD_BASE_GAP = AUC[0, :, 0].std(), (AUC[0, :, 0] - AUC[0, :, 1]).std()
log(f"one file: public AUC SD {SD_BASE_PUB:.5f}, public-minus-private SD {SD_BASE_GAP:.5f}; rule K = {K_RULE:.4f}")

In [ ]:
# Rebuild the public ladder files on the TEST set from the authors' public inputs (formulas copied from their notebooks),
# and measure how different consecutive rungs are (rank correlation). The last file is checked byte for byte against
# the published 0.95354 output (Areeb Ali's notebook asserts the same SHA-256).
TEST_FILES = {
    "A":   ("submission_095340.csv", "12b525b84f29e32d6511e9d6105f6dc5eac79ae41b10140eed3353ea57afc159"),   # Strike 10, amanatar
    "F":   ("submission_095330.csv", "f4a7a4f8636c99fabd82945c7d81956c5559c609dcb55cde6d4e03f80b0ea105"),   # Strike 9, Moran Z
    "LM":  (["lm_test.npy", "megalm_test_sha3168.npy"], "3168779be644a530c6bdc4c141cae168bfc2feb01ac21090c917cc978a7687d3"),
    "L7F": ("lm7_test_5fold.npy", "68639fec79f081e97d40185443fa3903c9db3e62698abb01dfe31dcee217462a"),
    "GPD": ("test_ours.npy", "96f3bdde816e604438e61eac9722f7a7402f1c39bb8efe8999793b23f91d02f2"),
    "MLP": ("test_gbn_bag5_w512.npy", "035d6544d7c03108fc42ee296a8660a45825e3b1b19b7cabce1c3f42cac4e7cb"),
    "GB":  ("test_gb_bag5.npy", "7d54e5f17e0707d88fec1a2cccb05885978ae6cc2a595c5f4575c7b7d45a9bb8"),
    "MS":  ("test.npy", "6c27aff7be744f7a0cc17809e0cd04fcdea0664234af88e3040a25235fb67906"),
    "G26": ("gpt2_s2026_test_f0.npy", "40e08058ed7632ee328c2fe7d97040b856b36d2636f32cc6d2781ec9db080695"),
}
T = {}
for k, (n, h) in TEST_FILES.items():
    p = find(n, h, required=False)
    if p is None: continue
    if str(p).endswith(".csv"):
        d = pd.read_csv(p); assert (d["id"].to_numpy() == sample["id"].to_numpy()).all(); T[k] = L(d["is_fraud"].to_numpy())
    else: T[k] = L(np.load(p, allow_pickle=False).astype(np.float64))
HAVE_LADDER = len(T) == len(TEST_FILES)
RUNGS = []   # (label, test-file score, reported public LB)
if HAVE_LADDER:
    A, F = T["A"], T["F"]
    a2 = A + 0.6 * (A - F)
    a3 = a2 + 0.12 * (T["LM"] - F)
    s352 = A + 0.6 * (A - F) + 0.09 * (T["LM"] - F) + 0.09 * (T["L7F"] - F) + 0.05 * (T["GPD"] - F) - 0.05 * (T["MLP"] - F) - 0.025 * (T["GB"] - F)
    s354 = L(s352) + 0.06 * (T["G26"] - T["MS"])
    s354b = L(s352) + 0.10 * (T["G26"] - T["MS"])
    out = (rankdata(s354) - 0.5) / NT
    chk = WORK / "submission.csv"; sample[["id"]].assign(is_fraud=out).to_csv(chk, index=False)
    print("rebuilt 0.95354 file identical to the published one:", sha256(chk) == "1e2e72fa26309ce630a19e0df780ae63d525901f33dc23f9058581ced5e9d6e0")
    chk.unlink()
    RUNGS = [("Strike 9 (Moran Z)", F, 95330), ("Strike 10 (amanatar)", A, 95340), ("+ 0.6 (S10 - S9)", a2, 95342),
             ("+ 0.12 megayak LM", a3, 95343), ("0.95352 file (several steps)", s352, 95352),
             ("+ 0.06 GPT-2 seed swap", s354, 95354)]
    rk = {lab: rankdata(s) for lab, s, _ in RUNGS}
    STEPS = []   # (label, public gain e-5, 1 - rho vs previous rung, predicted public noise SD e-5)
    for (l0, _, p0), (l1, _, p1) in zip(RUNGS[:-1], RUNGS[1:]):
        one_rho = 1 - np.corrcoef(rk[l0], rk[l1])[0, 1]
        STEPS.append((l1, p1 - p0, one_rho, K_RULE * np.sqrt(one_rho) * 1e5))
    one_rho = 1 - np.corrcoef(rankdata(s352), rankdata(s354b))[0, 1]
    STEPS_EXTRA = ("0.10 seed swap vs 0.95352 file", 2, one_rho, K_RULE * np.sqrt(one_rho) * 1e5)
    for s in STEPS: print(f"  {s[0]:32s} public {s[1]:+3d}e-5   1-rho {s[2]:.2e}   noise SD of the public gap ~{s[3]:.1f}e-5")
    del rk
else:
    print("ladder inputs not all attached; the rung figure uses the OOF moves only")
SIG_RUNG = float(np.median([s[3] for s in STEPS])) if HAVE_LADDER else float(np.median(SD_PUB))
log(f"typical noise of one rung on the public board: {SIG_RUNG:.1f}e-5")

In [ ]:
# Winner's curse, two ways.
# (1) Textbook: K probes with zero true gain, public noise SD sigma: the best public score is sigma * E[max of K normals].
KS = np.arange(1, 101)
_z = np.linspace(-6, 8, 4001)
_trap = getattr(np, "trapezoid", None) or np.trapz
EMAX = np.array([_trap(_z * k * norm.pdf(_z) * norm.cdf(_z) ** (k - 1), _z) for k in KS])
# (2) On real data: choose the best of k probes from the 'family' (and the ladder moves) by the public part of each
#     random split, then read the private part of the same split for the chosen probe.
rng = np.random.default_rng(7)
KGRID = [1, 2, 3, 5, 8, 12, 20, 30]
ALLI = np.arange(len(NAMES))
CURSE = {k: [] for k in KGRID}
for r in range(R):
    for k in KGRID:
        for _ in range(4):
            idx = rng.choice(ALLI, k, replace=False)
            j = idx[np.argmax(DELTA[idx, r, 0])]
            CURSE[k].append((DELTA[j, r, 0], DELTA[j, r, 1], TRUE[NAMES[j]]))
CURSE = {k: np.array(v) for k, v in CURSE.items()}
for k in KGRID:
    c = CURSE[k]; print(f"  best of {k:2d}: public {c[:, 0].mean():+5.1f}e-5   private {c[:, 1].mean():+5.1f}e-5   true (CV) {c[:, 2].mean():+5.1f}e-5")
# A frozen snapshot of the public leaderboard (2026-11-12 12:11 UTC): scores >= 0.95200 in units of 1e-5 above 0.95,
# and each team's number of submissions. Team names are left out on purpose.
LB_TOP = np.array([int(v) for v in """370,369,369,369,364,364,363,363,360,357,356,355,355,355,355,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,353,352,352,352,352,351,351,351,351,351,351,351,351,350,350,349,349,349,349,349,349,348,348,348,348,348,348,348,348,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,346,346,346,346,346,346,346,346,346,346,346,346,346,346,346,346,346,345,345,345,345,344,344,344,344,343,343,343,343,343,343,343,342,340,340,340,340,339,339,339,337,337,337,335,334,334,333,333,333,332,331,331,331,331,331,331,331,331,331,331,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,328,327,326,323,322,322,322,320,319,317,316,316,315,314,313,313,311,310,310,310,309,309,308,307,307,307,306,306,305,305,305,305,303,303,302,302,301,301,300,300,300,300,300,300,300,300,300,300,300,300,300,300,299,298,298,298,298,298,297,295,295,295,294,294,294,293,291,291,291,290,288,288,288,287,286,286,285,285,285,285,285,285,285,285,285,285,285,285,285,285,285,284,280,278,277,277,276,275,275,274,273,273,271,268,267,267,267,266,266,265,265,263,262,262,260,259,258,258,258,258,256,256,255,255,255,254,251,250,249,249,247,247,246,246,245,245,244,243,243,242,242,241,241,240,239,239,239,239,239,239,239,239,239,239,239,239,239,239,238,238,238,238,237,237,236,235,234,234,234,233,233,231,231,231,229,229,229,227,226,225,224,224,224,222,222,222,222,222,218,217,217,216,213,213,213,212,212,212,211,210,209,209,209,208,206,206,205,205,203,203,202,201,201,200,200,200""".split(",")])
LB_SUBS = np.array([int(v) for v in """101,31,111,102,10,55,93,18,111,2,85,20,18,20,1,96,60,10,42,96,20,11,20,10,104,17,10,13,5,2,118,13,58,20,11,84,48,91,17,66,47,17,27,18,7,48,6,10,51,39,24,15,12,4,90,9,65,10,71,4,14,95,19,30,6,57,66,5,18,8,7,1,26,5,11,34,9,2,7,14,24,64,7,11,7,18,4,10,36,10,43,15,17,6,8,7,12,6,102,45,10,27,16,1,14,8,1,20,12,38,8,7,5,15,13,22,27,42,8,120,11,42,15,12,7,23,73,73,10,12,11,30,29,10,23,19,98,78,22,31,6,2,13,11,8,16,5,8,10,2,11,7,2,11,7,7,4,15,80,27,21,2,13,8,2,6,9,8,9,5,11,10,15,3,3,7,30,40,5,33,5,4,14,26,13,5,13,46,18,23,7,10,28,28,12,16,51,1,9,30,79,6,2,23,17,11,6,30,38,31,21,51,36,22,47,6,25,16,5,3,4,4,2,1,19,10,2,3,28,19,16,1,32,2,8,77,30,16,6,5,6,5,21,31,40,3,39,4,34,10,5,26,10,2,9,11,3,2,10,7,12,5,2,9,1,3,14,3,69,14,19,15,58,3,4,120,16,100,11,10,4,3,5,7,1,9,27,79,5,12,4,78,14,3,9,4,12,5,1,8,6,16,3,5,9,20,38,1,6,5,33,5,9,5,5,25,12,10,17,1,59,6,4,6,12,2,16,2,9,26,4,36,55,31,16,3,9,9,25,15,6,19,1,5,10,4,8,11,24,5,4,30,5,16,32,7,33,39,5,1,1,1,16,9,13,4,7,7,9,7,1,9,19,3,4,1,3,11,3,4,3,7,25,5,13,7,30,31,6,41,7,7""".split(",")])
LB_TEAMS = 1577
TOP_SCORE = LB_TOP.max()
N_2SIG = int((LB_TOP >= TOP_SCORE - 2 * SIG_RUNG).sum())
log(f"leaderboard snapshot: {len(LB_TOP)} teams >= 0.95200; {N_2SIG} teams within 2 rung-noise SDs of #1")

<a id="hero"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">The whole story in one figure</div>

In [ ]:
# Hero figure: KPI tiles + four panels (rungs vs noise, public vs nested CV, winner's curse, the top band).
PUB_FRAUD = NPUB * y.mean()
MV = {m[0]: m for m in MOVES}
VARIANT = {"GPT-2 seed swap 0.10"}                               # the same rung at another weight: counted once
POS = [m[0] for m in MOVES if m[2] > 0 and m[0] not in VARIANT]  # distinct probes the public board rewarded
CONF = [n for n in POS if TRUE[n] > 0 and UP[n] >= 4]            # ... that nested CV also confirms (>= 4 of 5 folds)
SHORT = {"+0.12 megayak LM": "+ megayak LM", "+0.06 seed-7 LM": "+ seed-7 LM", "+0.12 public GPT-2": "+ public GPT-2",
         "+0.05 GP + DVAE": "+ GP + DVAE", "LM weight 0.12 -> 0.09": "LM 0.12 -> 0.09", "more of both LMs": "more of both LMs",
         "less seed-7 LM": "less seed-7 LM", "GPT-2 seed swap 0.06": "GPT-2 seed swap", "GPT-2 seed swap 0.10": "seed swap at 0.10"}

def hero_rungs(ax):
    steps = STEPS if HAVE_LADDER else [(n, MV[n][2], ONE_RHO[NAMES.index(n)], SD_PUB[NAMES.index(n)]) for n in POS]
    yy = np.arange(len(steps))[::-1]
    for yi, (lab, g, _, sd) in zip(yy, steps):
        ax.add_patch(FancyBboxPatch((-2 * sd, yi - 0.34), 4 * sd, 0.68, boxstyle="round,pad=0,rounding_size=0.12", facecolor=BAND, edgecolor="none", zorder=1))
        ax.add_patch(FancyBboxPatch((-sd, yi - 0.34), 2 * sd, 0.68, boxstyle="round,pad=0,rounding_size=0.12", facecolor="#d6d5d0", edgecolor="none", zorder=1))
        ax.barh(yi, g, height=0.36, color=PUB, zorder=3)
        ax.annotate(f"+{g}", (max(g, 2 * sd), yi), xytext=(6, 0), textcoords="offset points", va="center", fontsize=10, fontweight="bold", color=INK)
        ax.annotate(f"noise SD {sd:.1f}", (max(g, 2 * sd), yi), xytext=(30, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK2)
    ax.set_yticks(yy); ax.set_yticklabels([s[0] for s in steps], fontsize=9); ax.tick_params(axis="y", length=0)
    lim = max(max(s[1] for s in steps), 2 * max(s[3] for s in steps))
    ax.set_xlim(-lim * 1.05, lim * 2.0); ax.set_ylim(-0.6, len(steps) - 0.4)
    ax.xaxis.set_major_formatter(E5); ax.grid(axis="y", visible=False)
    ax.set_xlabel("public LB change of the step (x 1e-5)")
    zero_line(ax, "x")
    ax.legend(handles=[Patch(color=PUB, label="reported public gain"), Patch(color="#d6d5d0", label="1 SD of public noise"),
                       Patch(color=BAND, label="2 SD")], loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, fontsize=8.5)

def hero_scatter(ax):
    moves = [m[0] for m in MOVES]
    for n in moves:
        i = NAMES.index(n); x, yv = TRUE[n], MV[n][2]; good = n in CONF or (n in VARIANT and TRUE[n] > 0 and UP[n] >= 4)
        ax.plot([x, x], [yv - SD_PUB[i], yv + SD_PUB[i]], color=RULE, lw=2.2, solid_capstyle="round", zorder=2)
        ax.scatter(x, yv, s=110 if good else 60, color=CV if good else PUB, edgecolor=SURF, lw=1.5, zorder=4)
    ax.set_xlim(-9.5, 4); ax.set_ylim(-16.5, 6)
    ax.plot([-9.5, 4], [-9.5, 4], color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
    ax.text(-9.3, -10.0, "dashed: public = CV", color=MUTED, fontsize=8.5, va="top")
    zero_line(ax, "x"); zero_line(ax, "y")
    ax.annotate("+ public GPT-2: -14 public,\nand down on 5 of 5 folds", (TRUE["+0.12 public GPT-2"], -14), xytext=(14, 0), textcoords="offset points",
                fontsize=8.5, color=INK2, va="center")
    ax.annotate("+ megayak LM", (TRUE["+0.12 megayak LM"], MV["+0.12 megayak LM"][2]), xytext=(-8, 9), textcoords="offset points",
                fontsize=8.5, color=INK2, ha="right")
    ax.annotate("GPT-2 seed swap:\nup on both boards", (TRUE["GPT-2 seed swap 0.06"], 2), xytext=(4, 34), textcoords="offset points",
                fontsize=9, color=CVTXT, fontweight="bold", ha="center", arrowprops=dict(arrowstyle="-", color=CVTXT, lw=1))
    cl = [n for n in moves if n not in ("+0.12 public GPT-2", "+0.12 megayak LM", "GPT-2 seed swap 0.06", "GPT-2 seed swap 0.10")]
    x0, x1 = min(TRUE[n] for n in cl) - 0.35, max(TRUE[n] for n in cl) + 0.35
    y0, y1 = min(MV[n][2] for n in cl) - 0.8, max(MV[n][2] for n in cl) + 0.8
    ax.add_patch(FancyBboxPatch((x0, y0), x1 - x0, y1 - y0, boxstyle="round,pad=0,rounding_size=0.4", facecolor="none",
                                edgecolor=PUB, lw=1, ls=(0, (2, 2)), zorder=3))
    ax.annotate(f"{len(cl)} more probes: public {min(MV[n][2] for n in cl):+d} to {max(MV[n][2] for n in cl):+d},\n"
                f"nested CV {min(TRUE[n] for n in cl):+.1f} to {max(TRUE[n] for n in cl):+.1f}", (x0, y0 + 0.2), xytext=(-10, -26),
                textcoords="offset points", fontsize=8.5, color=INK2, ha="right", va="top", arrowprops=dict(arrowstyle="-", color=PUB, lw=0.8))
    ax.xaxis.set_major_formatter(E5); ax.yaxis.set_major_formatter(E5)
    ax.set_xlabel("change in nested CV on the public OOF (x 1e-5)"); ax.set_ylabel("reported public LB change (x 1e-5)")
    ax.text(0.99, 0.02, "grey bars: 1 SD of public noise", transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color=MUTED)

def hero_curse(ax):
    ax.plot(KS, SIG_RUNG * EMAX, color=PUB, lw=2, label="best public gain of K null probes (theory)")
    ax.fill_between(KS, 0, SIG_RUNG * EMAX, color=PUB, alpha=0.07, lw=0)
    kk = np.array(KGRID)
    ax.plot(kk, [CURSE[k][:, 0].mean() for k in KGRID], "o", color=PUB, ms=7, mec=SURF, mew=1.5, label="best of K real probes: public part")
    ax.plot(kk, [CURSE[k][:, 1].mean() for k in KGRID], "o-", color=PRIV, ms=7, mec=SURF, mew=1.5, lw=1.4, label="the same pick: private part")
    zero_line(ax)
    ax.set_ylim(min(-4, min(CURSE[k][:, 1].mean() for k in KGRID) - 2), SIG_RUNG * EMAX[-1] * 1.12)
    med = int(np.median(LB_SUBS[:50]))
    ax.axvline(med, color=MUTED, lw=1, ls=(0, (3, 3)))
    ax.text(med * 1.07, SIG_RUNG * EMAX[-1] * 0.45, f"median top-50 team:\n{med} submissions", fontsize=8.5, color=INK2, va="top")
    ax.set_xscale("log"); ax.set_xticks([1, 2, 5, 10, 20, 50, 100]); ax.set_xticklabels(["1", "2", "5", "10", "20", "50", "100"])
    ax.yaxis.set_major_formatter(E5); ax.yaxis.set_major_locator(MaxNLocator(6, integer=True))
    ax.set_xlabel("number of probes K compared on the public board")
    ax.set_ylabel("score change of the chosen probe (x 1e-5)"); ax.legend(loc="upper left", fontsize=8.5)

def hero_board(ax):
    lo = 95300
    vals, cnt = np.unique(LB_TOP[LB_TOP + 95000 >= lo], return_counts=True)
    ax.axvspan(TOP_SCORE - 2 * SIG_RUNG, TOP_SCORE + 0.6, color=BAND, zorder=0, lw=0)
    ax.text(TOP_SCORE - 2 * SIG_RUNG + 0.4, cnt.max() * 1.22, f"within 2 rung-noise\nSDs of #1: {N_2SIG} teams", fontsize=8.5, color=INK2, va="top")
    ax.bar(vals, cnt, width=0.75, color=[PUB if c < 10 else NAVY for c in cnt], zorder=3)
    big = [(v, c) for v, c in zip(vals, cnt) if c >= 15]
    for j, (v, c) in enumerate(big):
        near = any(abs(v - v2) <= 1 and c2 > c for v2, c2 in big)
        ax.text(v - (1.2 if near else 0), c + 0.6, f"0.95{v:03d}\n{c} teams", ha="right" if near else "center", va="bottom", fontsize=8, color=INK)
    ax.set_xlim(lo - 95000 - 1, TOP_SCORE + 1); ax.set_ylim(0, cnt.max() * 1.3)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"0.95{int(v):03d}")); ax.xaxis.set_major_locator(MaxNLocator(7, integer=True))
    ax.set_xlabel("public LB score"); ax.set_ylabel("teams with exactly this score")

fig = plt.figure(figsize=(16, 12.4))
fig.text(0.02, 0.975, "How much of the public top is real?", fontsize=24, fontweight="bold", color=INK, va="top")
fig.text(0.02, 0.937, "Ladder steps of one or two points sit inside the noise of the 60k-row public board; nested CV confirms few of them, "
         "and the best of many public probes looks better than it is.", fontsize=12, color=INK2, va="top")
tw, gap, ty = 0.2275, 0.0133, 0.815
TILES = [("Public part of the test set", f"~{NPUB / 1000:.0f}k rows", f"about {PUB_FRAUD:,.0f} frauds at the train rate", PUB),
         ("Noise of one ladder rung", f"±{SIG_RUNG:.1f}e-5", "1 SD of the public gap between consecutive rungs", NAVY),
         ("Public gains confirmed by nested CV", f"{len(CONF)} of {len(POS)}", "public probes that also gain on >= 4 of 5 folds", CV),
         ("Teams within 2 rung SDs of #1", f"{N_2SIG}", f"of {LB_TEAMS:,} on the board (snapshot 12 Nov)", PRIV)]
for i, t_ in enumerate(TILES): tile(fig, [0.02 + i * (tw + gap), ty, tw, 0.095], *t_)
axs = [fig.add_axes([0.205, 0.465, 0.29, 0.26]), fig.add_axes([0.605, 0.43, 0.37, 0.295]),
       fig.add_axes([0.06, 0.06, 0.40, 0.28]), fig.add_axes([0.56, 0.06, 0.41, 0.28])]
hero_rungs(axs[0]);   panel_title(axs[0], "A  Each rung gains about one noise SD", "public gain of consecutive ladder files vs. the noise of that gap")
hero_scatter(axs[1]); panel_title(axs[1], "B  Public says up; nested CV mostly says flat or down", "every public probe with OOF files behind it, replayed on the wide-stack OOF")
hero_curse(axs[2]);   panel_title(axs[2], "C  The best of many probes gains on public, not on private", "random public/private draws from the OOF; probes = small steps towards public members")
hero_board(axs[3]);   panel_title(axs[3], "D  The top band is a few rung-noise SDs wide", "public leaderboard snapshot; tall bars = many teams at exactly the score of a public notebook")
plt.show()

<div style="background: #eef4fc; border-left: 5px solid #2a78d6; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>In one paragraph.</b> A ladder rung of one or two points is about one noise SD of the public gap between the two
files (panel A). Replayed on a nested OOF, most rungs that won on public are flat or negative; the one that holds up,
a second seed of the same model, has an obvious mechanism (panel B). Choosing the best of many probes by public score
guarantees a public gain even when nothing is better, and the private part does not repeat it (panel C). The top of
the board is only a few such noise SDs wide, and many teams sit on the exact score of a public notebook (panel D).
</div>

<a id="s1"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">1 &middot; The competition in five minutes</div>

* **Data.** 699,960 synthetic card transactions to train on, 299,984 to predict; 24 columns (amount, timing, device,
  merchant, risk scores, flags). About **1.67%** of the training rows are fraud.
* **Metric.** ROC-AUC: the probability that a random fraud is ranked above a random legitimate transaction. Only the
  **order** of the predictions matters, which is why every public blend here works on logit ranks
  `L(v) = logit((rank(v) - 0.5) / n)`.
* **The split.** The public leaderboard scores **20%** of the test rows, the private leaderboard (the final ranking)
  the other **80%**. Both are fixed for everyone; nobody knows which rows are which.
* **Units.** Gaps at the top are a few hundred-thousandths of AUC. We write them as **points of 1e-5**:
  0.95354 - 0.95352 = +2 points.

In [ ]:
# The two boards in rows and expected frauds (train fraud rate applied to the test set).
fig, ax = plt.subplots(figsize=(13, 2.9))
rate = y.mean()
segs = [("train", 0, N, MUTED, f"train  {N:,} rows  |  {int(y.sum()):,} frauds"),
        ("public", 0, NPUB, PUB, f"public LB  ~{NPUB:,} rows\n~{NPUB * rate:,.0f} frauds"),
        ("private", NPUB, NT - NPUB, PRIV, f"private LB  ~{NT - NPUB:,} rows  |  ~{(NT - NPUB) * rate:,.0f} frauds")]
ax.barh(1, N, color="#e6e5df", height=0.55); ax.text(N / 2, 1, segs[0][4], ha="center", va="center", fontsize=10.5, color=INK)
ax.barh(0, NPUB, color=PUB, height=0.55); ax.barh(0, NT - NPUB, left=NPUB + 1500, color=PRIV, height=0.55)
ax.text(NPUB / 2, 0, segs[1][4], ha="center", va="center", fontsize=9.5, color="white", fontweight="bold")
ax.text(NPUB + (NT - NPUB) / 2, 0, segs[2][4], ha="center", va="center", fontsize=10.5, color="white", fontweight="bold")
ax.set_yticks([1, 0]); ax.set_yticklabels(["train", "test"], fontsize=11); ax.tick_params(axis="y", length=0)
ax.set_xlim(0, N * 1.01); ax.grid(False); ax.spines["left"].set_visible(False)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:.0f}k")); ax.set_xlabel("rows")
headline(fig, f"The public board ranks everyone on about {NPUB * rate:,.0f} frauds",
         "Public = 20% of the test rows, private = the other 80%. Frauds are estimated from the training fraud rate.")
plt.show()

<a id="s2"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">2 &middot; How noisy is the public LB?</div>

Two different questions hide behind "the public LB is noisy":

* **How far is one file's public score from its private score?** Far: the AUC of a fixed file on ~60k rows with about
  a thousand frauds wanders by a few *thousandths*. But every team is scored on the same rows, so this mostly shifts
  the whole board together.
* **How reliable is the public gap between two files?** This is what decides rankings and ladder steps. Two
  files that rank almost every row the same way share most of their noise, so the gap is far less noisy &mdash; but
  the gaps at the top are also tiny.

We measure both on the nested OOF of the public wide LM stack: draw a public part (~60k rows) and a private part
(~240k rows) at random, score every file on both, repeat a few hundred times.

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))
gap1 = (AUC[0, :, 0] - AUC[0, :, 1]) * 1e5
ax.hist(gap1, bins=30, color=MUTED, edgecolor=SURF)
ax.set_xlabel("public AUC minus private AUC of ONE file (x 1e-5)"); ax.set_ylabel("random splits"); ax.xaxis.set_major_formatter(E5)
panel_title(ax, f"One file: SD {np.std(gap1):.0f} points", "the whole board shares this shift")
iS = NAMES.index("GPT-2 seed swap 0.06")
bins_ = np.linspace(min(DELTA[iS, :, 0].min(), DELTA[iS, :, 1].min()), max(DELTA[iS, :, 0].max(), DELTA[iS, :, 1].max()), 36)
ax2.hist(DELTA[iS, :, 0], bins=bins_, color=PUB, edgecolor=SURF, alpha=0.9, label="public gap")
ax2.hist(DELTA[iS, :, 1], bins=bins_, color=PRIV, edgecolor=SURF, alpha=0.75, label="private gap")
ax2.axvline(TRUE["GPT-2 seed swap 0.06"], color=INK, lw=1.5)
ax2.text(TRUE["GPT-2 seed swap 0.06"] + 0.3, ax2.get_ylim()[1] * 0.95, f"nested CV {TRUE['GPT-2 seed swap 0.06']:+.1f}", fontsize=9, va="top")
zero_line(ax2, "x"); ax2.xaxis.set_major_formatter(E5); ax2.legend(loc="upper left", fontsize=9)
ax2.set_xlabel("gap between two consecutive ladder files (x 1e-5)")
panel_title(ax2, f"Two near-identical files: public SD {SD_PUB[iS]:.1f}, private SD {SD_PRV[iS]:.1f}",
            f"the GPT-2 seed swap: public gap < 0 in {np.mean(DELTA[iS, :, 0] < 0):.0%} of splits")
headline(fig, f"One public score swings by +-{np.std(gap1):.0f} points; the gap between two near-identical files by +-{SD_PUB[iS]:.1f}, still more than a rung",
         "Independent random public (~60k rows) and private (~240k rows) draws from the public wide-stack OOF")
plt.show()

### A one-line rule for the noise of a public gap
For two files `a` and `b`, the noise of their public gap depends almost only on **how differently they rank the rows**,
that is on `1 - rho`, where `rho` is the rank (Spearman) correlation of the two files. Over every move measured here the
relation is a clean square root:

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.2))
order_ = np.argsort(ONE_RHO)
is_move = np.array([n in MV or n == "tree move" for n in NAMES])
ax.scatter(ONE_RHO[~is_move], SD_PUB[~is_move], s=36, color=PUB, alpha=0.55, edgecolor="none", label="random probe steps (public part)")
ax.scatter(ONE_RHO[is_move], SD_PUB[is_move], s=60, color=PUB, edgecolor=SURF, lw=1.2, label="public ladder probes (public part)")
ax.scatter(ONE_RHO, SD_PRV, s=22, color=PRIV, alpha=0.6, edgecolor="none", label="the same steps, private part")
xx = np.logspace(np.log10(ONE_RHO.min() * 0.6), np.log10(ONE_RHO.max() * 1.6), 50)
ax.plot(xx, K_RULE * np.sqrt(xx) * 1e5, color=NAVY, lw=2)
ax.plot(xx, K_RULE * np.sqrt(xx) * 1e5 * np.sqrt(NPUB / (NT - NPUB)), color=PRIV, lw=1.5, ls=(0, (4, 3)))
ax.text(xx[-1], K_RULE * np.sqrt(xx[-1]) * 1e5, f"  public: {K_RULE:.4f} x sqrt(1 - rho)", va="center", fontsize=10, fontweight="bold", color=NAVY)
ax.text(xx[-1], K_RULE * np.sqrt(xx[-1]) * 1e5 * 0.5, "  private: half of it", va="center", fontsize=9.5, color=PRIV)
if HAVE_LADDER:
    for lab, g, orho, sd in STEPS:
        ax.axvline(orho, color=RULE, lw=1, zorder=0)
    ax.text(max(s[2] for s in STEPS) * 1.08, ax.get_ylim()[0] * 1.15, "grey lines: consecutive\npublic ladder files", fontsize=8.5, color=INK2, va="bottom")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("1 - rank correlation of the two files"); ax.set_ylabel("SD of the gap (x 1e-5)")
ax.set_xlim(xx[0], xx[-1] * 4.5); ax.legend(loc="upper left", fontsize=9)
headline(fig, f"Noise of a public gap = {K_RULE:.4f} x sqrt(1 - rho): about {K_RULE * np.sqrt(1e-4) * 1e5:.0f} points when rho = 0.9999",
         "Each dot: SD of the gap between the wide-stack OOF and one small move of it, over random 20/80 splits")
plt.show()

In [ ]:
# Shown: use the rule on your own two submission files (the constant K_RULE was fitted above on public OOF; it assumes
# a test set and fraud rate like this competition's, and two files of similar strength).
def public_gap_noise(file_a, file_b, k=K_RULE):
    """1 SD (in AUC) of the public-LB gap between two submissions, from their rank correlation alone."""
    a = pd.read_csv(file_a)["is_fraud"].to_numpy(); b = pd.read_csv(file_b)["is_fraud"].to_numpy()
    rho = np.corrcoef(rankdata(a), rankdata(b))[0, 1]
    return k * np.sqrt(max(1 - rho, 0)), rho
for r_ in (0.99999, 0.9999, 0.9995, 0.999, 0.995):
    print(f"rho {r_:<8}  public-gap noise ~ {K_RULE * np.sqrt(1 - r_) * 1e5:5.1f} points   private ~ {K_RULE * np.sqrt(1 - r_) * 1e5 / 2:4.1f}")

<div style="background: #eef4fc; border-left: 5px solid #2a78d6; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>How to read it.</b> Two files with rank correlation 0.9999 have a public gap noise of a few points, about the size
of a typical ladder step. A gain of one or two points between such files is <b>not evidence</b> of anything: the same
change would show a loss on public in a large share of possible splits. The private gap is half as noisy (four times
more rows), so the private board will not simply repeat the public order.
</div>

<a id="s3"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">3 &middot; Anatomy of a ladder</div>

A ladder starts from a strong public file and adds small steps in logit-rank space, keeping each step that the public
score rewards. The current public top (0.95354) is built like this, and its authors published **every rung and every
probe** (goodpjw2008's *LM Ladder*, Areeb Ali's *Three Ladders*):

```
s352  = A + 0.6 (A - F) + 0.09 (LM - F) + 0.09 (L7F - F) + 0.05 (GPD - F) - 0.05 (MLP - F) - 0.025 (GB - F)   # 0.95352
score = L(s352) + 0.06 (G26 - MS)                                                                            # 0.95354
```

`A` = Strike 10 (amanatar, 0.95340), `F` = Strike 9 (Moran Z, 0.95330), `LM` = megayak's target-first LM, `L7F` =
goodpjw2008's seed-7 LM, `GPD` = daoviet's GP + DVAE, `MLP`/`GB` = two OOF-bank bags, `MS` = masayakawamata's
target-first GPT-2, `G26` = a second seed of it (goodpjw2008). The notebook rebuilt this file above from the public
inputs and checked it byte for byte against the published one.

**The part no one can check.** `A` and `F` publish no OOF predictions, so the base and the extrapolation
`0.6 (A - F)` (move further along the direction from Strike 9 to Strike 10) have no out-of-fold check at all. Every
other ingredient has public OOF files, so each step can be replayed on a nested OOF, as the ladder authors did
themselves. The table collects all of them.

In [ ]:
rows = []
for n, d, pub, src in MOVES:
    i = NAMES.index(n)
    rows.append({"probe (step applied to the ladder file)": n, "public LB change": f"{pub:+d}",
                 "nested CV change": f"{TRUE[n]:+.1f}", "folds up": f"{UP[n]}/5",
                 "public noise SD": f"{SD_PUB[i]:.1f}", "P(public sign = private sign)": f"{AGREE[i]:.0%}", "source of the public number": src})
i = NAMES.index("tree move")
rows.append({"probe (step applied to the ladder file)": "full tree move (-0.10 MLP, -0.05 GB)", "public LB change": "part of v15",
             "nested CV change": f"{TRUE['tree move']:+.1f}", "folds up": f"{UP['tree move']}/5", "public noise SD": f"{SD_PUB[i]:.1f}",
             "P(public sign = private sign)": f"{AGREE[i]:.0%}", "source of the public number": "Areeb Ali v15 (with other changes)"})
TAB = pd.DataFrame(rows)
print("All changes in points of 1e-5. Nested CV = applied to the nested OOF of the public wide LM stack (CV %.6f)." % CV_W)
TAB.style.hide(axis="index") if hasattr(TAB.style, "hide") else TAB

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.6))
names_ = [m[0] for m in MOVES] + ["tree move"]
pubv = [m[2] for m in MOVES] + [np.nan]
cvv = [TRUE[n] for n in names_]
sdv = [SD_PUB[NAMES.index(n)] for n in names_]
yy = np.arange(len(names_))[::-1]
for yi, s in zip(yy, sdv):
    ax.add_patch(FancyBboxPatch((-s, yi - 0.38), 2 * s, 0.76, boxstyle="round,pad=0,rounding_size=0.1", facecolor=BAND, edgecolor="none", zorder=0))
ax.barh(yy + 0.17, pubv, height=0.32, color=PUB, label="reported public LB change", zorder=3)
ax.barh(yy - 0.17, cvv, height=0.32, color=[CV if (c > 0 and UP[n] >= 4) else "#9ccfba" if c > 0 else PRIV for c, n in zip(cvv, names_)],
        label="nested CV change", zorder=3)
for yi, n, c in zip(yy, names_, cvv):
    ax.text(max(0, c) + 0.4 if c >= 0 else c - 0.4, yi - 0.17, f"{c:+.1f}  ({UP[n]}/5 folds)", va="center",
            ha="left" if c >= 0 else "right", fontsize=8.5, color=INK2)
ax.set_yticks(yy); ax.set_yticklabels([SHORT.get(n, "full tree move") for n in names_], fontsize=9.5); ax.tick_params(axis="y", length=0)
zero_line(ax, "x"); ax.xaxis.set_major_formatter(E5); ax.set_xlabel("change (x 1e-5)"); ax.grid(axis="y", visible=False)
ax.set_xlim(-17, 8)
ax.legend(handles=[Patch(color=PUB, label="reported public LB change"), Patch(color=CV, label="nested CV: up on >= 4 of 5 folds"),
                   Patch(color="#9ccfba", label="nested CV: up on < 4 folds"), Patch(color=PRIV, label="nested CV: down"), Patch(color=BAND, label="+-1 SD public noise of the step")],
          loc="lower left", fontsize=9)
headline(fig, f"Only the GPT-2 seed swap gains on both boards; {len(POS) - len(CONF)} other public gains are not confirmed by nested CV",
         "Every public probe that has OOF files behind it, replayed on the nested OOF of the public wide LM stack")
plt.show()

**Reading the table.** The ladder authors saw the same thing and said so: Areeb Ali writes that *"every earlier ladder
step lost on CV"* and that the two-seed rung is the first one that holds up; goodpjw2008 calls a rung *"a bet, not a
measurement"*. Our replay agrees with their own CV checks: the two-seed rung gains about +1.9e-5 on 5 of 5 folds
(goodpjw2008: +1.9e-5) and the full tree move loses about 7.7e-5 (Areeb Ali: -7.4e-5).

The two-seed rung is also the one with a clear mechanism: averaging two seeds of the same model removes part of one
seed's noise, which is a gain one expects on any rows. The other steps tune *how much* of a member the file holds, and
there the public board and CV disagree; with a public noise SD of 3&ndash;7 points per step, the public board cannot
settle a one- or two-point question.

<a id="s4"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">4 &middot; The winner's curse</div>

Try `K` probes, keep the one with the best public score. Even if none of them is truly better, the best of `K` noisy
numbers is above zero, by about `sigma x E[max of K standard normals]`: about 0.8 sigma for K = 3, 1.5 sigma for K = 10,
2.0 sigma for K = 30, 2.5 sigma for K = 100. The private part of the same split does not share that luck, so the pick
falls back. This is the textbook *winner's curse* (selection bias) and, repeated over many rounds of public feedback,
*adaptive overfitting* of the public holdout.

Below, the probes are real: small steps from the wide stack towards (or away from) each OOF-bank member and each public
LM. On every random split we pick the best of `K` by its public gap and read its private gap.

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1.15, 1]))
kk = np.array(KGRID)
pub_m = np.array([CURSE[k][:, 0].mean() for k in KGRID]); prv_m = np.array([CURSE[k][:, 1].mean() for k in KGRID])
tru_m = np.array([CURSE[k][:, 2].mean() for k in KGRID])
ax.fill_between(kk, prv_m, pub_m, color=PUB, alpha=0.08, lw=0)
ax.plot(kk, pub_m, "o-", color=PUB, ms=7, mec=SURF, label="public gap of the pick")
ax.plot(kk, prv_m, "o-", color=PRIV, ms=7, mec=SURF, label="private gap of the same pick")
ax.plot(kk, tru_m, "o--", color=CV, ms=6, mec=SURF, lw=1.4, label="nested CV of the pick (its true size)")
ax.annotate(f"{pub_m[-1] - prv_m[-1]:.1f} points\nlost on private", xy=(kk[-1], (pub_m[-1] + prv_m[-1]) / 2), xytext=(-12, 0),
            textcoords="offset points", ha="right", va="center", fontsize=9.5, fontweight="bold", color=INK)
zero_line(ax); ax.set_xscale("log"); ax.set_xticks(KGRID); ax.set_xticklabels([str(k) for k in KGRID])
ax.yaxis.set_major_formatter(E5); ax.set_xlabel("probes compared on public (K)"); ax.set_ylabel("gap vs. the base file (x 1e-5)")
ax.legend(loc="upper left", fontsize=9)
panel_title(ax, "The more probes you compare, the more the pick is luck", "average over random 20/80 splits")
c20 = CURSE[20]
ax2.scatter(c20[:, 0], c20[:, 1], s=8, color=PUB, alpha=0.25, edgecolor="none")
lim = np.percentile(np.abs(c20[:, :2]), 99.5)
ax2.plot([-lim, lim], [-lim, lim], color=MUTED, lw=1, ls=(0, (4, 3)))
zero_line(ax2); zero_line(ax2, "x"); ax2.set_xlim(-lim, lim); ax2.set_ylim(-lim, lim)
ax2.xaxis.set_major_formatter(E5); ax2.yaxis.set_major_formatter(E5)
ax2.set_xlabel("public gap of the best of 20 (x 1e-5)"); ax2.set_ylabel("its private gap (x 1e-5)")
ax2.text(0.97, 0.05, f"public > 0 in {np.mean(c20[:, 0] > 0):.0%} of splits\nprivate > 0 in {np.mean(c20[:, 1] > 0):.0%}",
         transform=ax2.transAxes, ha="right", va="bottom", fontsize=9.5, color=INK)
panel_title(ax2, "Best of 20: almost always up on public", "each dot = one random split")
headline(fig, "Keeping the best public probe buys public points that the private board does not pay",
         "Real probes on the public wide-stack OOF; public and private parts drawn independently in each split")
plt.show()

<div style="background: #fff4ec; border-left: 5px solid #eb6834; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>How many probes are behind a public file?</b> The version tables list ten versions for one ladder (v9&ndash;v20) and
about twenty scored probes for the other, and these ladders build on each other. On the leaderboard snapshot the
median top-50 team has made about 20 submissions, and the top teams over 100. With a rung noise of a few points,
the theory says the best of 20 null probes shows about 1.9 noise SDs of public "gain". That is the size of the whole
visible spread between many of the top files.
</div>

<a id="s5"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">5 &middot; CV can be tuned too</div>

Nested CV is the referee in this notebook, so it is fair to ask how a CV number can itself be inflated. Three common
ways, each measured below on public data:

1. **Early stopping on the scored fold.** The fold that reports the score also picks the number of trees.
2. **Selecting on the reported folds.** A blend, a member set or a weight chosen on the same OOF rows it is scored on.
3. **A test inside one fold.** An inner CV run on the rows of a single outer fold measures something else.

In [ ]:
# 5.1  Early stopping on the scored fold: one 4-leaf LightGBM per outer fold on the raw columns (learning rate 0.1),
# trained on 90% of the training part. We record the AUC every 10 trees on the scored fold AND on the held-out 10%,
# so the two ways of choosing the number of trees are compared on the very same model.
import lightgbm as lgb
NTH = int(os.environ.get("OMP_NUM_THREADS", 4))
X = train.drop(columns=["id", "is_fraud"]).copy()
for c in X.columns:
    if X[c].dtype == object: X[c] = X[c].astype("category")
    elif X[c].dtype == bool: X[c] = X[c].astype(int)
ES_FOLDS, ES_ROUNDS, STEP = [0, 1, 2], 1500, 10
ES = []
for k in ES_FOLDS:
    va = FOLDS[k]; tr = np.setdiff1d(np.arange(N), va)
    inner = np.random.default_rng(k).random(len(tr)) < 0.10
    a, h = tr[~inner], tr[inner]
    bst = lgb.train(dict(objective="binary", num_leaves=4, learning_rate=0.1, min_data_in_leaf=100, verbose=-1,
                         num_threads=NTH, seed=k), lgb.Dataset(X.iloc[a], y[a]), num_boost_round=ES_ROUNDS)
    zh, zv = np.zeros(len(h)), np.zeros(len(va)); its, ah, av = [], [], []
    for s in range(0, ES_ROUNDS, STEP):
        zh += bst.predict(X.iloc[h], start_iteration=s, num_iteration=STEP, raw_score=True)
        zv += bst.predict(X.iloc[va], start_iteration=s, num_iteration=STEP, raw_score=True)
        its.append(s + STEP); ah.append(roc_auc_score(y[h], zh)); av.append(roc_auc_score(y[va], zv))
    its, ah, av = map(np.array, (its, ah, av))
    i_sc, i_in = av.argmax(), ah.argmax()
    ES.append(dict(fold=k, its=its, av=av, i_sc=i_sc, i_in=i_in, opt=(av[i_sc] - av[i_in]) * 1e5))
    log(f"fold {k}: trees picked on scored fold {its[i_sc]} (AUC {av[i_sc]:.6f}), on inner 10% {its[i_in]} "
        f"(scored AUC {av[i_in]:.6f}) -> optimism {ES[-1]['opt']:+.1f}e-5")
del X

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw=dict(width_ratios=[1.7, 1]))
FC = [C[0], C[6], C[3]]
for e, col in zip(ES, FC):
    rel = (e["av"] - e["av"][e["i_sc"]]) * 1e5
    ax.plot(e["its"], rel, color=col, lw=1.6, label=f"fold {e['fold']}")
    ax.plot(e["its"][e["i_sc"]], 0, "o", ms=8, color=col, mec=SURF, mew=1.5, zorder=5)
    ax.plot(e["its"][e["i_in"]], rel[e["i_in"]], "D", ms=7, mfc=SURF, mec=col, mew=1.8, zorder=5)
low = min((e["av"][e["i_in"]] - e["av"][e["i_sc"]]) * 1e5 for e in ES)
ax.set_ylim(low * 1.6 - 5, 8); ax.set_xlim(100, ES_ROUNDS); ax.yaxis.set_major_formatter(E5)
ax.set_xlabel("number of trees"); ax.set_ylabel("scored-fold AUC minus its maximum (x 1e-5)")
ax.legend(handles=[Line2D([], [], color=c, lw=2, label=f"fold {e['fold']}") for e, c in zip(ES, FC)] +
          [Line2D([], [], marker="o", ls="", color=INK2, label="trees chosen on the scored fold"),
           Line2D([], [], marker="D", ls="", mfc=SURF, mec=INK2, label="trees chosen on a held-out 10%")], loc="lower right", fontsize=8.5)
panel_title(ax, "The top of a noisy curve is always the highest point", "scored-fold AUC along training, 4-leaf LightGBM, learning rate 0.1")
opts = [e["opt"] for e in ES]
ax2.bar(range(len(ES)), opts, color=PRIV, width=0.55)
for i, v in enumerate(opts): ax2.text(i, v + 0.3, f"+{v:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
ax2.set_xticks(range(len(ES))); ax2.set_xticklabels([f"fold {e['fold']}" for e in ES]); ax2.yaxis.set_major_formatter(E5)
ax2.set_ylabel("optimism of the reported fold AUC (x 1e-5)"); ax2.set_ylim(0, max(opts) * 1.3 + 1)
panel_title(ax2, f"Free optimism: {np.mean(opts):+.1f}e-5 per fold", "scored-fold pick minus held-out pick")
headline(fig, f"Early stopping on the scored fold reports {np.mean(opts):.0f} points more than an honest choice of trees",
         "Same model, two ways to choose the number of trees: on the fold that is scored, or on a held-out 10% of the training part")
plt.show()

The held-out 10% is itself a noisy judge, so its pick lands away from the scored-fold peak: that is the point. The
scored-fold pick hides this noise and reports the peak; any choice made without the scored rows pays for it.
A fast learner (learning rate 0.1) is the bad case: its curve is jagged, so the maximum of the scored fold sits well
above what a choice made elsewhere achieves. With the slower learning rate most public tree members use (0.03) a
separate measurement of the same comparison gave about **+2 to +3e-5 per member**. Per member that is small next to
fold noise, but it is **systematic**: a stack of a hundred members
that all stopped on their scored fold inherits it, and stack weights then favour the members whose OOF is most
optimistic. Choosing the number of trees on a held-out slice of the training part
costs nothing and removes it.

In [ ]:
# 5.2  Selecting on the reported folds: greedy hill climbing (Caruana) over the OOF bank's 20 core members.
# Non-nested: choose the blend on all OOF rows, report its CV on the same rows.  Nested: choose on 4 folds, score the 5th.
names = list(BANK); M = np.stack([BANK[n] for n in names], 1).astype(np.float32)
HC_STEPS = 10
def hill_climb(rows):
    """Greedy forward selection with replacement on `rows`; returns the list of picks (indices into names)."""
    yr = y[rows]; Mr = M[rows]
    first = int(np.argmax([auc_sort(yr, Mr[:, j]) for j in range(M.shape[1])]))
    picks, s = [first], Mr[:, first].astype(np.float64).copy()
    for _ in range(HC_STEPS - 1):
        sc = [auc_sort(yr, s + Mr[:, j]) for j in range(M.shape[1])]
        j = int(np.argmax(sc)); picks.append(j); s += Mr[:, j]
    return picks
def blend(picks, upto): return M[:, picks[:upto]].mean(1)
full_picks = hill_climb(np.arange(N))
HC_NON = [cv(blend(full_picks, t)) for t in range(1, HC_STEPS + 1)]
HC_NEST = np.zeros((5, HC_STEPS))
for k, b in enumerate(FOLDS):
    p = hill_climb(np.setdiff1d(np.arange(N), b))
    for t in range(1, HC_STEPS + 1): HC_NEST[k, t - 1] = roc_auc_score(y[b], blend(p, t)[b])
    log(f"  hill climbing, outer fold {k}: picks {[names[j] for j in p[:4]]} ...")
HC_NEST_CV = HC_NEST.mean(0)
BEST_SINGLE = max(cv(M[:, j]) for j in range(M.shape[1]))
log(f"hill climbing: reported (non-nested) CV {HC_NON[-1]:.6f}, nested CV {HC_NEST_CV[-1]:.6f}, gap {(HC_NON[-1] - HC_NEST_CV[-1]) * 1e5:+.1f}e-5")

In [ ]:
# 5.3  A test inside one fold: does adding megayak's LM (already a member of the wide stack) to W help?
# Outer test: weights fitted on the other four folds, scored on fold k.  Inner test: a 5-fold CV inside fold k's rows only.
Z = np.c_[W, O["LM"]]
INOUT = []
for k, b in enumerate(FOLDS):
    rest = np.setdiff1d(np.arange(N), b)
    lr_ = LogisticRegression(C=1.0, max_iter=500).fit(Z[rest], y[rest])
    outer = roc_auc_score(y[b], Z[b] @ lr_.coef_[0]) - roc_auc_score(y[b], W[b])
    ins = []
    for it, ib in StratifiedKFold(5, shuffle=True, random_state=0).split(b, y[b]):
        l2 = LogisticRegression(C=1.0, max_iter=500).fit(Z[b[it]], y[b[it]])
        ins.append(roc_auc_score(y[b[ib]], Z[b[ib]] @ l2.coef_[0]) - roc_auc_score(y[b[ib]], W[b[ib]]))
    INOUT.append((k, np.mean(ins) * 1e5, outer * 1e5))
    print(f"  fold {k}: inner test {np.mean(ins) * 1e5:+6.1f}e-5   outer test {outer * 1e5:+6.1f}e-5")

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1.25, 1]))
t = np.arange(1, HC_STEPS + 1)
ax.plot(t, (np.array(HC_NON) - BEST_SINGLE) * 1e5, color=PRIV, marker="o", ms=6, mec=SURF, label="reported: chosen and scored on the same OOF")
ax.plot(t, (HC_NEST_CV - BEST_SINGLE) * 1e5, color=CV, marker="o", ms=6, mec=SURF, label="nested: chosen on 4 folds, scored on the 5th")
gap = (HC_NON[-1] - HC_NEST_CV[-1]) * 1e5
ax.annotate(f"gap {gap:+.1f}e-5", xy=(HC_STEPS, (HC_NON[-1] - BEST_SINGLE) * 1e5), xytext=(-8, 14), textcoords="offset points",
            ha="right", fontsize=10, fontweight="bold", color=INK)
ax.set_xlabel("hill-climbing step (members in the blend, with repeats)"); ax.set_ylabel("CV gain over the best single member (x 1e-5)")
ax.yaxis.set_major_formatter(E5); ax.set_xticks(t); ax.legend(loc="lower right", fontsize=9)
panel_title(ax, f"Selection on the scored rows reads {gap:.0f} points high", "greedy blend of the OOF bank's 20 core members")
ks = [r[0] for r in INOUT]
for k, i_, o_ in INOUT:
    ax2.plot([i_, o_], [k, k], color=RULE, lw=3, zorder=1)
ax2.scatter([r[1] for r in INOUT], ks, s=70, color=PRIV, zorder=3, label="inner CV inside the fold", edgecolor=SURF, lw=1.5)
ax2.scatter([r[2] for r in INOUT], ks, s=70, color=CV, zorder=3, label="outer test on the fold", edgecolor=SURF, lw=1.5)
zero_line(ax2, "x"); ax2.set_yticks(ks); ax2.set_yticklabels([f"fold {k}" for k in ks]); ax2.invert_yaxis()
ax2.xaxis.set_major_formatter(E5); ax2.set_xlabel("gain from adding a member already in the stack (x 1e-5)")
ax2.legend(loc="lower right", fontsize=9)
n_dis = sum(np.sign(i_) != np.sign(o_) for _, i_, o_ in INOUT)
panel_title(ax2, f"Inner and outer verdicts disagree on {n_dis} of 5 folds", "logistic re-weighting of the wide stack + megayak LM")
headline(fig, "Where you select and where you score must be different rows",
         "Left: a blend picked on the OOF it is scored on. Right: a decision taken inside one fold versus across folds.")
plt.show()

<div style="background: #fff4ec; border-left: 5px solid #eb6834; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Rules of thumb for an honest CV.</b> (1) Stop trees on a held-out slice of the training part, never on the scored
fold. (2) Every choice a pipeline makes (members, weights, C, gates, transforms) is made inside the training part of
each outer fold: that is <i>nested</i> CV. (3) Decide across all outer folds, not inside one: a fold-local test sees
that fold's own calibration and noise. (4) A stack fitted once on the full OOF and then reported on the same OOF reads
high; fit it again fold by fold to get the number you can compare.
</div>

<a id="s6"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">6 &middot; What did not work</div>

Negative results are rarely published and save everyone time. First, public probes that the ladder authors reported
as losses (all credit to them for publishing these):

| probe on a ladder file | public change | reported by |
|---|---|---|
| + 0.12 of the public target-first GPT-2 (already inside every ladder file) | -14 | goodpjw2008 (v1 probes) |
| + 5% / + 2% TabPFN-3.5 | -1 / -1 | Areeb Ali (v18 and probes) |
| 10% mix with another public stack | -1 | Areeb Ali |
| + 10% / + 20% of a GPT-2 + GLM fusion file | 0 / 0 | Areeb Ali |
| more of both LMs (+0.04, +0.03) | -2 | Areeb Ali |
| an extra LM step on daoviet's file, other LM kept at 0.12 | 0 | goodpjw2008 |

Areeb Ali's summary: *"new model families (TabPFN, GBDT files, other stacks) lose a point"*. On nested CV, adding more
language-model weight to a wide stack also loses (-2.8e-5, up on only 2 of 5 folds in our check), even though public
ladders gained points from LM steps.

Second, "smarter" ways to combine a bank of public OOF predictions, each compared on **nested CV over 5 outer folds**
with every setting fixed in advance, against simply keeping a strong logistic stack:

| method | nested CV vs. the plain stack | folds better |
|---|---|---|
| greedy hill climbing (Caruana), bank that already contains the stack | +0.0 (it picks the stack alone) | 0/5 |
| logistic regression on all ~80 deduplicated members, C by inner CV | -27.7e-5 | 0/5 |
| weights proportional to pairwise AUC | -40.6e-5 | 0/5 |
| two experts gated by a data segment | -30.9e-5 | 0/5 |
| stack + a small segment-gate correction | -1.9e-5 | 0/5 |
| non-linear meta-model (4-leaf LightGBM on members, member disagreement, raw columns) | -39e-5 to -56e-5 | 0&ndash;1/5 |

<div style="background: #eefaf4; border-left: 5px solid #1baf7a; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Pattern.</b> Once a linear stack holds the strong members, cleverer combiners mostly fit noise: they lose on
every outer fold by far more than any ladder step gains. The one kind of change that kept gaining on nested CV in the
public record is <b>new information</b> (another seed or another view of a strong model), not new weights.
</div>

<a id="s7"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">7 &middot; What to expect on private, and how to pick two final files</div>

Put the pieces together for the current public top file. We replay its checkable steps on the nested OOF, in the
order they were built, and set them beside the public points each one earned.

In [ ]:
D_a3 = 0.12 * (O["LM"] - W)
D_352 = 0.09 * (O["LM"] - W) + 0.09 * (O["L7F"] - W) + 0.05 * (O["GPD"] - W) - 0.05 * (O["MLP"] - W) - 0.025 * (O["GB"] - W)
c_a3, c_352 = W + D_a3, W + D_352
c_354 = L(c_352) + 0.06 * (O["G26"] - O["MS"])
CHAIN = [("Strike 9 -> Strike 10", 10, None, None), ("+ 0.6 (S10 - S9)", 2, None, None),
         ("+ 0.12 megayak LM", 1, (cv(c_a3) - CV_W) * 1e5, folds_up(c_a3, W)),
         ("-> 0.95352 file\n(4 steps)", 9, (cv(c_352) - cv(c_a3)) * 1e5, folds_up(c_352, c_a3)),
         ("+ GPT-2 seed swap", 2, (cv(c_354) - cv(c_352)) * 1e5, folds_up(c_354, c_352))]
CHECK_PUB = sum(c[1] for c in CHAIN if c[2] is not None); CHECK_CV = sum(c[2] for c in CHAIN if c[2] is not None)
for lab, p, c, u in CHAIN:
    print(f"  {lab.replace(chr(10), ' '):28s} public {p:+3d}   nested CV " + ("  no OOF" if c is None else f"{c:+5.1f}  ({u}/5 folds up)"))
print(f"checkable steps: public {CHECK_PUB:+d} points, nested CV {CHECK_CV:+.1f} points")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.0))
x = np.arange(len(CHAIN))
ax.bar(x - 0.2, [c[1] for c in CHAIN], width=0.38, color=PUB, label="public LB points earned")
for i, (lab, p, c, u) in enumerate(CHAIN):
    ax.text(i - 0.2, p + 0.3, f"+{p}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
    if c is None:
        ax.bar(i + 0.2, 4, width=0.38, color="none", edgecolor=MUTED, hatch="///", lw=0)
        ax.text(i + 0.2, 4.3, "no OOF:\nuntestable", ha="center", va="bottom", fontsize=8.5, color=INK2)
    else:
        ax.bar(i + 0.2, c, width=0.38, color=CV if (c > 0 and u >= 4) else PRIV)
        ax.text(i + 0.2, c + (0.3 if c >= 0 else -0.3), f"{c:+.1f}\n({u}/5)", ha="center", va="bottom" if c >= 0 else "top", fontsize=9, color=INK)
zero_line(ax); ax.set_xticks(x); ax.set_xticklabels([c[0] for c in CHAIN], fontsize=9.5); ax.yaxis.set_major_formatter(E5)
ax.set_ylabel("points (x 1e-5)"); ax.grid(axis="x", visible=False)
ax.set_ylim(min(-8, min(c[2] for c in CHAIN if c[2] is not None) - 3), 14)
ax.legend(handles=[Patch(color=PUB, label="public LB points earned"), Patch(color=CV, label="nested CV: confirmed"),
                   Patch(color=PRIV, label="nested CV: not confirmed"), Patch(facecolor="none", edgecolor=MUTED, hatch="///", label="no OOF published")],
          loc="upper right", fontsize=9)
headline(fig, f"Of the +24 public points from Strike 9 to the top file, the checkable +{CHECK_PUB} are worth {CHECK_CV:+.1f} on nested CV",
         "Each step of the 0.95354 ladder file replayed on the nested OOF of the public wide LM stack (moves applied in the same order)")
plt.show()

**What this means for the private board** (an estimate, not a forecast):

* The absolute private score of every file can sit a few *thousandths* away from its public score, in the same
  direction for everyone. Nobody can know that direction; it does not change the ranking.
* The **ranking** inside the top band depends on gaps of a few points, and each gap carries a public noise of about
  one ladder-rung SD. Expect the top of the private board to reshuffle by that much.
* Steps that nested CV did not confirm have an expected private value near their CV value, not their public value.
  Steps without any OOF (the base files and the extrapolation) are a genuine unknown: they may hold, or not.
* The two-seed rung is the exception that proves the method: it won on both boards and has a mechanism.

<div style="background: #eefaf4; border-left: 5px solid #1baf7a; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Picking two final files &mdash; general advice.</b><br>
<b>1. One CV-backed file.</b> Every choice in it (members, weights, early stopping) made inside the training folds; its
nested CV is the number you trust. Its public score may sit below the ladders; that is fine.<br>
<b>2. One different hypothesis.</b> If you believe a public ladder's untestable base carries real signal, that is a
legitimate second pick &mdash; but make the second file genuinely <i>different</i> from the first (lower rank
correlation), so the two picks hedge each other.<br>
<b>3. Never two near-copies.</b> Two files at rank correlation 0.9999 land within a few points of each other on
private: the second slot is wasted.<br>
<b>4. Do not let the public board choose between close files.</b> If two candidates differ by less than about two
noise SDs (use <code>public_gap_noise()</code> above), the public board cannot tell them apart; decide by CV or by
reasoning.
</div>

<a id="s8"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">8 &middot; Full credits and licences</div>

**Notebooks whose published numbers are quoted** (public scores, probe tables, formulas):

* goodpjw2008 &mdash; [S6E11 LM Ladder | Two GPT-2 Seeds | LB 0.95354](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-two-gpt-2-seeds-lb-0-95354),
  [LM Ladder Part 1](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-part-1-own-lm-step-lb-0-95346),
  [Part 12 | Wide Stack + 2nd LM](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298)
  (the wide LM stack and its nested OOF), and the `find()` helper and logit-rank convention used here.
* Areeb Ali &mdash; [S6E11 | Three Ladders, Two Seeds | LB 0.95354](https://www.kaggle.com/code/areebbb/s6e11-three-ladders-two-seeds-lb-0-95354)
  and [Three Ladders, Two Steps](https://www.kaggle.com/code/areebbb/s6e11-three-ladders-two-steps-lb-0-95347): version history, probe table, CV-evidence table.
* amanatar &mdash; [S6E11 | Podium Apex | Public LB 0.95340](https://www.kaggle.com/code/amanatar/s6e11-podium-apex-public-lb-0-95340) (Strike 10).
* Moran Z &mdash; [S6E11 | Grandmaster Ladder | LB 0.95330](https://www.kaggle.com/code/moranzavdi/s6e11-grandmaster-ladder-lb-0-95330) (Strike 9).
* masayakawamata &mdash; [GPT-2 Target-First Classifier](https://www.kaggle.com/code/masayakawamata/s6e11-gpt-2-target-first-classifier-cv-0-9500); with Matapanino, the target-first GPT-2.
* megayak &mdash; [per-value digit TE shallow-tree stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack) and the target-first LM.
* samanyu1808 &mdash; the LM-step idea on these files (as credited by goodpjw2008 and Areeb Ali).
* Chris Deotte &mdash; [S6E10 1st place write-up](https://www.kaggle.com/competitions/playground-series-s6e10/writeups/1st-place-tfms-and-llms-with-tabular-data), the target-first language-model idea.

**Datasets attached to this notebook** (each file checked by SHA-256):

| dataset | files used | licence |
|---|---|---|
| [goodpjw2008/s6e11-fraud-series-artifacts](https://www.kaggle.com/datasets/goodpjw2008/s6e11-fraud-series-artifacts) | `wide_lm_stack_nested.npy` | CC0-1.0 |
| [goodpjw2008/s6e11-oof-bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank) | 20 core OOF members, MLP/GB bags (OOF + test), `manifest.csv` | CC0-1.0 |
| [goodpjw2008/s6e11-compact-lm-seed7](https://www.kaggle.com/datasets/goodpjw2008/s6e11-compact-lm-seed7) | seed-7 LM, OOF + test | CC0-1.0 |
| [goodpjw2008/s6e11-target-first-gpt2-seed-2026](https://www.kaggle.com/datasets/goodpjw2008/s6e11-target-first-gpt2-seed-2026) | GPT-2 seed 2026, OOF + test | CC0-1.0 |
| [megayak/s6e11-target-first-lm](https://www.kaggle.com/datasets/megayak/s6e11-target-first-lm) | target-first LM, OOF + test | CC0-1.0 |
| [masayakawamata/s6e11-gpt2-target-first-oof](https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof) | target-first GPT-2, OOF + test | CC BY 4.0 |
| [daoviet/s6e11-gp-dvae-orthogonal-apex](https://www.kaggle.com/datasets/daoviet/s6e11-gp-dvae-orthogonal-apex) | GP + DVAE, OOF + test | CC0-1.0 |
| [amanatar/s6e11-strike10-podium-apex-095340](https://www.kaggle.com/datasets/amanatar/s6e11-strike10-podium-apex-095340) | `submission_095340.csv` | CC0-1.0 |
| [moranzavdi/s6e11-strike9-lb-095330](https://www.kaggle.com/datasets/moranzavdi/s6e11-strike9-lb-095330) | `submission_095330.csv` | CC0-1.0 |

The competition data belongs to Kaggle's Playground Series S6E11 (competition rules apply). The leaderboard snapshot
(scores and submission counts only, 2026-11-12 12:11 UTC) is from the public leaderboard page. This notebook's code is
released under Apache-2.0. Its `submission.csv` is the public 0.95354 ladder file, rebuilt from the inputs above; the credit for it goes to the ladder authors.

Thank you again to everyone above. If this notebook helped you, please upvote the source notebooks too, starting with
goodpjw2008 and Areeb Ali.

In [ ]:
log("done")